In [ ]:
!pip install transformers

In [ ]:
!pip install "transformers[torch]"

In [ ]:
import pandas as pd
from transformers import T5Tokenizer,Trainer,TrainingArguments,T5ForConditionalGeneration

In [ ]:
train_data=pd.read_csv("samsum-train.csv")

In [ ]:
val_data=pd.read_csv("samsum-validation.csv")

In [ ]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [ ]:
val_data.head()

In [ ]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [ ]:
train_data.sample(10)

,id,dialogue,summary
2104,13820891,"Ann: <file_photo>\r\nLucy: Awwww, how cute!!\r...",Ann's daughter is 2 years old. She has a pink ...
10529,13681259,Mark: What’s happening about the heating?\r\nJ...,The radiators and timings don't work properly....
12129,13716042,Alice: DIY beauty products? Yes or no?\r\nElla...,Ella considers DYI products better for sensiti...
5427,13862386,Jarvis: Thank you very much for the Gifticon y...,Jarvis had a good time with his family. He is ...
3238,13864468,Daniel: Look who's dancing!\nDaniel: <file_vid...,Daniel was dancing.
14209,13680396,"Daria: <file_photo>\r\nLiz: OMG, congrats! You...",Daria can now drive a car.
2155,13820504,Haley: anybodys seen the new plebs?\r\nAston: ...,Hall has seen the new season of the Plebs. Peg...
14109,13828563,Noah: hello\r\nTom: hi! what's up?\r\nNoah: yo...,Noah is looking for a gift for his friend. His...
6328,13728267,Everett: have you see tristan lately?\r\nEmily...,Everett and Emily want to get in touch with Tr...
8750,13681144,Claire: hi honey\r\nMark: hi\r\nClaire: what a...,"Claire wants Mark to come to her today, beacus..."


In [ ]:
train_data.shape

(14732, 3)

In [ ]:
val_data.shape

(818, 3)

In [ ]:
# Random Sampling

train_data=train_data.sample(n=4000,random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=500,random_state=42).reset_index(drop=True)

In [ ]:
train_data.shape,val_data.shape

((4000, 3), (500, 3))

## Data Preprocessing

In [ ]:
import re

def clean_data(text):
  text=re.sub(r"\r\n"," ",text) # lines
  text=re.sub(r"s+"," ",text) # spaces
  text=re.sub(r"<.*?>","",text) #html tags
  text=text.strip().lower()

  return text

In [ ]:
train_data["dialogue"]=train_data["dialogue"].apply(clean_data)
train_data["summary"]=train_data["summary"].apply(clean_data)

val_data["dialogue"]=val_data["dialogue"].apply(clean_data)
val_data["summary"]=val_data["summary"].apply(clean_data)

In [ ]:
train_data.head()

,id,dialogue,summary
0,13811908,violet: hi! i came acro thi au tin' article...,violet ent claire au tin' article.
1,13716431,pat: so doe anyone know when the tream i go...,pat and lou are waiting for the tream but kev...
2,13810214,jane: jane: whaddya think? shona: thi ur ti...,jane i updating her tinder profile tonight an...
3,13729823,"adam: do u have a map of pari ? tom: ye , why?...",tom ha a map of pari .
4,13681400,"frank: hi, how' the family? mike: great! sam'...","mike i happy, becau e sam' moved out. mike a..."


## Tokenize

In [ ]:
tokenizer=T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# raw data=> tokenized inputs for fine-tuning

def tokenize(data):
  inputs=tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
  targets=tokenizer(data["summary"],padding="max_length",max_length=150,truncation=True)

  inputs["labels"]=targets["input_ids"] # token ids =>add to input as labels
  return inputs

In [ ]:
train_dataset=train_data.apply(tokenize,axis=1).tolist()
val_dataset=val_data.apply(tokenize,axis=1).tolist()

In [ ]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 3, 9, 2771, 3, 7436, 185, 3, 17, 77, 31, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1413, 15, 3, 1222, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2763, 3, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2763, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [ ]:
# input ids-dialogue -> token ids
# attention mask
# labels - target ->summary token

# 1 => EOS(End of Sequence)
# 0 => padding

In [ ]:
len(train_dataset[0]["input_ids"])

512

In [ ]:
len(train_dataset[0]["labels"])

150

In [ ]:
len(train_dataset[0]["attention_mask"])

512

In [ ]:
# If attention_mask==0:
#    valid ids
# else: attention_mask==0
#   invalid ids => padding values


In [ ]:
type(train_dataset),type(val_dataset)

(list, list)

## Building our Model

In [ ]:
model=T5ForConditionalGeneration.from_pretrained("t5-small")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

### Fine-Tuning

In [ ]:
import torch

if torch.backends.mps.is_available():
  device=torch.device("mps")
elif torch.cuda.is_available():
  device=torch.device("cuda")
else:
  device=torch.device("cpu")

print("device:",device)
model.to(device)

In [ ]:
# Training Arguments

training_args=TrainingArguments(

    output_dir="./results",
    num_train_epochs=6,
    weight_decay=0.01,
    per_device_eval_batch_size=8,
    per_device_train_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500

)

In [ ]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset

)

In [ ]:
# Training the model

trainer.train()

Epoch,Training Loss,Validation Loss
1,3.436326,0.516488
2,0.528022,0.469247
3,0.488924,0.454098
4,0.471714,0.447558
5,0.460139,0.443953


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,3.436326,0.516488
2,0.528022,0.469247
3,0.488924,0.454098
4,0.471714,0.447558
5,0.460139,0.443953
6,0.455427,0.442476


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9734252370198567, metrics={'train_runtime': 1360.6901, 'train_samples_per_second': 17.638, 'train_steps_per_second': 2.205, 'total_flos': 3248203235328000.0, 'train_loss': 0.9734252370198567, 'epoch': 6.0})

In [ ]:
model.save_pretrained("./saved_summarizer_model")
tokenizer.save_pretrained("./saved_summarizer_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summarizer_model/tokenizer_config.json',
 './saved_summarizer_model/tokenizer.json')

In [ ]:
model=T5ForConditionalGeneration.from_pretrained("./saved_summarizer_model")
tokenizer=T5Tokenizer.from_pretrained("./saved_summarizer_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## Test the core logic for Summarization

In [ ]:
def summarize_dialogue(dialogue):
  dialogue=clean_data(dialogue) # clean

  # Tokenize
  inputs=tokenizer(
      dialogue,
      padding="max_length",
      max_length=512,
      truncation=True,
      return_tensors="pt",
  ).to(device)

  # Generate the Summary =>token ids
  model.to(device)
  targets=model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_length=150,
    num_beams=4,
    early_stopping=True
  )

  # Decoding the Token ids =>decoded our output to summary

  summary=tokenizer.decode(targets[0],skip_special_tokens=True)
  return summary

In [ ]:
test_dialogue="""	Ollie: Hi , are you in Warsaw Jane: yes, just back! Btw are you free for diner the 19th? Ollie: nope! Jane: and the 18th? Ollie: nope, we have this party and you must be there, remember? Jane: oh right! i lost my calendar.. thanks for reminding me Ollie: we have lunch this week? Jane: with pleasure! Ollie: friday? Jane: ok Jane: what do you mean " we don't have any more whisky!" lol.. Ollie: what!!! Jane: you just call me and the all thing i heard was that sentence about whisky... what's wrong with you? Ollie: oh oh... very strange! i have to be carefull may be there is some spy in my mobile! lol Jane: dont' worry, we'll check on friday. Ollie: don't forget to bring some sun with you Jane: I can't wait to be in Morocco.. Ollie: enjoy and see you friday Jane: sorry Ollie, i'm very busy, i won't have time for lunch tomorrow, but may be at 6pm after my courses?this trip to Morocco was so nice, but time consuming! Ollie: ok for tea! Jane: I'm on my way.. Ollie: tea is ready, did you bring the pastries? Jane: I already ate them all... see you in a minute Ollie: ok"""


In [ ]:
summary=summarize_dialogue(test_dialogue)
print("Summary:",summary)

Summary: <pad> jane and ollie are in war aw. jane will be in morocco on friday.</s>
